# CS-Jail-UR: Stage C (judge gates) + Exp 2 judging, CPU only

Runs headless with **Save Version → Save & Run All (Commit)**. No GPU: the
judge is the OpenAI API, and Exp 2 responses are judged from the cache.

Before saving:
1. Settings → Accelerator **None**, Internet **On**.
2. Add-ons → Secrets → `OPENAI_API_KEY` (the paid key) with **Attached** ticked.
3. **Add Input** → your *private* dataset made from `stageC_upload.zip`.
4. Set the flags in the first code cell.

Order: harm gate (~1 h), benign gate if its folder was uploaded (~20 min),
then, **only if the harm gate PASSES**, Exp 2 main judging (~7 h at
30,000 tokens/min; stopped at 9 h so the run always finishes and saves).
Download `stageC_outputs.zip` from the version's **Output** tab. Keep this
notebook private: outputs contain prompts, responses and judge rationales.

In [ ]:
# Set these before "Save & Run All".
RUN_HARM_GATE = True
RUN_BENIGN_GATE = True          # skipped automatically if the benign folder was not uploaded
JUDGE_EXP2_MAIN = True          # runs only if the harm gate PASSES (~7 h)
JUDGE_EXP2_ROBUSTNESS = False   # ~4.5 h; use a second version so the 12 h limit is never hit

import os
import pathlib
import shutil
from IPython import get_ipython


def run(cmd):
    """Run a shell command with streamed output; return its exit code."""
    print("$", cmd, flush=True)
    get_ipython().system(cmd)
    return get_ipython().user_ns.get("_exit_code", 0)


def sh(cmd):
    """Like run(), but stop the whole notebook if the command fails."""
    code = run(cmd)
    if code:
        raise RuntimeError(f"step failed (exit {code}): {cmd}")

In [ ]:
# Repo + Python 3.12 environment (judge and test packages only; no GPU stack).
%cd /kaggle/working
if not os.path.isdir("NLP_Research"):
    sh("git clone --branch revision/final-692-dataset https://github.com/mshamoonbutt/NLP_Research.git")
%cd /kaggle/working/NLP_Research
sh("git pull --ff-only && git log --oneline -1")
sh("python -m pip install -q uv")
sh("python -m uv venv --allow-existing --python 3.12 /root/py312")
sh('python -m uv pip install --python /root/py312/bin/python -e ".[judge,dev]"')
os.environ["PATH"] = "/root/py312/bin:" + os.environ["PATH"]
os.environ["VIRTUAL_ENV"] = "/root/py312"
os.environ.pop("PYTHONPATH", None)
sh("which python && python --version")

In [ ]:
# Secret, uploaded inputs, and a check that the judge is the frozen one.
from kaggle_secrets import UserSecretsClient
os.environ["OPENAI_API_KEY"] = UserSecretsClient().get_secret("OPENAI_API_KEY")
marker = next(pathlib.Path("/kaggle/input").rglob("CSJAIL_STAGEC_UPLOAD"))
shutil.copytree(marker.parent / "outputs", "outputs", dirs_exist_ok=True)
sh("ls outputs/exp1 outputs/exp2/main outputs/exp2/robustness")
pathlib.Path("/tmp/check_judge.py").write_text("""
from csjail.judge import load_judge_config
c = load_judge_config()
fp = c.fingerprint("harm")["fingerprint_id"]
print("judge", c.model_id, "| harm fingerprint", fp)
assert fp == "b0d6676cf5d89d08", "judge config differs from the frozen one"
""")
sh("python /tmp/check_judge.py")

In [ ]:
# Harm gate on the merged 1,200-item sample (720 + 480 top-up).
harm = None
if RUN_HARM_GATE:
    harm = run("python scripts/calibrate_judge.py --sample-dir outputs/exp1/validation-kaggle-01-merged "
               "--development-sample-dir outputs/exp1/development-cpu-20260926")
    print("HARM GATE EXIT:", harm, "(0 PASS, 1 FAIL, 2 INSUFFICIENT_EVIDENCE, 3 input error)")

In [ ]:
# Benign (over-refusal) gate, if its merged folder was uploaded.
benign = None
if RUN_BENIGN_GATE and os.path.isdir("outputs/exp1/benign-validation-merged"):
    benign = run("python scripts/calibrate_judge.py --kind benign "
                 "--sample-dir outputs/exp1/benign-validation-merged")
    print("BENIGN GATE EXIT:", benign, "(0 PASS, 1 FAIL, 2 INSUFFICIENT_EVIDENCE, 3 input error)")
elif RUN_BENIGN_GATE:
    print("Benign gate skipped: outputs/exp1/benign-validation-merged was not uploaded")

In [ ]:
# Exp 2 judging from the cache (no model is loaded). Only after a harm PASS.
if JUDGE_EXP2_MAIN and harm == 0:
    code = run("timeout 9h python -m csjail.run_eval --out-dir outputs/exp2/main --judge-only")
    print("EXP 2 MAIN JUDGING EXIT:", code, "(124 = stopped at 9 h; upload judgments.jsonl to resume)")
elif JUDGE_EXP2_MAIN:
    print("Exp 2 main judging skipped: the harm gate did not PASS in this run")
if JUDGE_EXP2_ROBUSTNESS and os.path.exists("outputs/exp1/judge_validation_manifest.json"):
    code = run("timeout 9h python scripts/exp2_robustness.py --judge-only --greedy-results outputs/exp2/main")
    print("EXP 2 ROBUSTNESS JUDGING EXIT:", code)

In [ ]:
# Package everything for one download from the Output tab.
keep = [p for p in ("outputs/exp1/judge_validation_manifest.json",
                    "outputs/exp1/judge_validation_manifest_benign.json",
                    "outputs/exp1/validation-kaggle-01-merged", "outputs/exp1/benign-validation-merged",
                    "outputs/exp2") if os.path.exists(p)]
sh("rm -f /kaggle/working/stageC_outputs.zip && zip -qr /kaggle/working/stageC_outputs.zip " + " ".join(keep))
sh("ls -lh /kaggle/working/stageC_outputs.zip")
print("DONE | harm gate:", harm, "| benign gate:", benign)